# CO3133 - Text-Image Classification with CrisisMMD

This notebook implements the full **text-image classification** pipeline for BTL1.

## Dataset description
- **Dataset:** CrisisMMD v2.0
- **Source:** official CrisisNLP release for multimodal disaster analysis
- **Input:** one tweet text paired with one disaster image
- **Task:** humanitarian categories classification
- **Model comparison:** **CLIP** vs **VisualBERT**

## What does "humanitarian categories" mean?
Each tweet-image pair is assigned to one of five response-oriented categories:
1. `affected_individuals`
2. `infrastructure_and_utility_damage`
3. `rescue_volunteering_or_donation_effort`
4. `other_relevant_information`
5. `not_humanitarian`

These labels capture whether the post contains actionable disaster-response information.

## What does "agreed-label split" mean?
CrisisMMD provides multiple annotation files. This notebook uses the **official agreed-label split**,
which keeps examples where annotators agreed on the label and already defines the `train`, `dev`, and `test`
splits. That gives a cleaner and more reproducible evaluation setup.


In [ ]:
from pathlib import Path
import json
import math
import random
import tarfile
import urllib.request
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, AutoTokenizer, CLIPModel, CLIPVisionModel, VisualBertModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")


def find_btl1_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if base.name == "btl1" and (base / "data").exists() and (base / "artifacts").exists():
            return base
        candidate = base / "btl1"
        if candidate.exists() and (candidate / "data").exists() and (candidate / "artifacts").exists():
            return candidate
    raise FileNotFoundError("Could not locate the btl1 directory. Run the notebook from the repo root or from inside btl1/.")


BTL1_ROOT = find_btl1_root()
REPO_ROOT = BTL1_ROOT.parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"BTL1 root: {BTL1_ROOT}")
print(f"Device: {DEVICE}")


In [ ]:
MM_DIR = BTL1_ROOT / "data" / "multimodal" / "crisismmd"
EXT_DIR = MM_DIR / "external"
RAW_DIR = MM_DIR / "raw"
PROCESSED_DIR = MM_DIR / "processed"
ARTIFACT_DIR = BTL1_ROOT / "artifacts" / "multimodal"

DATASET_URL = "https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz"
AGREED_SPLIT_URL = "https://crisisnlp.qcri.org/data/crisismmd/crisismmd_datasplit_agreed_label.zip"
ALL_SPLIT_URL = "https://crisisnlp.qcri.org/data/crisismmd/crisismmd_datasplit_all.zip"
PAPER_URL = "https://arxiv.org/abs/1805.00713"
OFFICIAL_PAGE = "https://crisisnlp.qcri.org/crisismmd"

DATASET_ARCHIVE = EXT_DIR / "CrisisMMD_v2.0.tar.gz"
AGREED_ARCHIVE = EXT_DIR / "crisismmd_datasplit_agreed_label.zip"
ALL_SPLIT_ARCHIVE = EXT_DIR / "crisismmd_datasplit_all.zip"

LABELS = [
    "affected_individuals",
    "infrastructure_and_utility_damage",
    "rescue_volunteering_or_donation_effort",
    "other_relevant_information",
    "not_humanitarian",
]
label2id = {label: idx for idx, label in enumerate(LABELS)}
EXPECTED_COLUMNS = ["event_name", "tweet_id", "image_id", "tweet_text", "image", "label", "label_text", "label_image", "label_text_image"]

SEED = 42
MAX_ROWS_PER_SPLIT = None  # Set an integer for quick debugging if needed.
DOWNLOAD_IF_MISSING = True

for folder in [EXT_DIR, RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Dataset archive: {DATASET_ARCHIVE}")
print(f"Agreed split archive: {AGREED_ARCHIVE}")
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Artifact dir: {ARTIFACT_DIR}")


In [ ]:
CLIP_CFG = {
    "model_name": "openai/clip-vit-base-patch32",
    "batch_size": 16,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "epochs": 3,
    "max_length": 77,
    "dropout": 0.2,
}

VB_CFG = {
    "model_name": "uclanlp/visualbert-vqa-coco-pre",
    "vision_name": "openai/clip-vit-base-patch32",
    "batch_size": 8,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "epochs": 3,
    "max_length": 64,
    "dropout": 0.2,
}

PATIENCE = 1

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
def download_if_missing(url: str, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        print(f"Already exists: {target}")
        return target
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(f"Missing file and auto-download is disabled: {target}")
    print(f"Downloading {url} -> {target}")
    urllib.request.urlretrieve(url, target)
    return target


def extract_tar_if_needed(archive_path: Path, destination: Path, marker_name: str):
    marker = destination / marker_name
    if marker.exists():
        print(f"Skipping tar extraction: {archive_path.name}")
        return
    if not archive_path.exists():
        raise FileNotFoundError(f"Missing tar archive: {archive_path}")
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as handle:
        handle.extractall(destination)
    marker.write_text("ok", encoding="utf-8")


def extract_zip_if_needed(archive_path: Path, destination: Path, marker_name: str):
    marker = destination / marker_name
    if marker.exists():
        print(f"Skipping zip extraction: {archive_path.name}")
        return
    if not archive_path.exists():
        raise FileNotFoundError(f"Missing zip archive: {archive_path}")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "r") as handle:
        handle.extractall(destination)
    marker.write_text("ok", encoding="utf-8")


def find_one(root: Path, name: str, is_dir: bool = False) -> Path:
    matches = sorted(path for path in root.rglob(name) if path.is_dir() == is_dir and "__MACOSX" not in path.parts)
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {root}")
    return matches[0]


download_if_missing(DATASET_URL, DATASET_ARCHIVE)
download_if_missing(AGREED_SPLIT_URL, AGREED_ARCHIVE)
download_if_missing(ALL_SPLIT_URL, ALL_SPLIT_ARCHIVE)

extract_tar_if_needed(DATASET_ARCHIVE, RAW_DIR, ".crisismmd_dataset_extracted")
extract_zip_if_needed(AGREED_ARCHIVE, EXT_DIR / "agreed_label", ".crisismmd_agreed_label_extracted")

DATASET_DIR = find_one(RAW_DIR, "CrisisMMD_v2.0", is_dir=True)
AGREED_DIR = find_one(EXT_DIR / "agreed_label", "crisismmd_datasplit_agreed_label", is_dir=True)
SPLIT_FILES = {
    "train": find_one(AGREED_DIR, "task_humanitarian_text_img_agreed_lab_train.tsv"),
    "dev": find_one(AGREED_DIR, "task_humanitarian_text_img_agreed_lab_dev.tsv"),
    "test": find_one(AGREED_DIR, "task_humanitarian_text_img_agreed_lab_test.tsv"),
}

print(f"Dataset dir: {DATASET_DIR}")
print(f"Agreed split dir: {AGREED_DIR}")
print({name: str(path) for name, path in SPLIT_FILES.items()})


In [ ]:
def resolve_image_path(relative_path: str) -> Path:
    candidate = DATASET_DIR / Path(str(relative_path))
    if candidate.exists():
        return candidate
    fallback = DATASET_DIR / "data_image" / Path(str(relative_path)).name
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f"Image file not found for {relative_path}")


def prepare_split(split_path: Path, split_name: str) -> pd.DataFrame:
    frame = pd.read_csv(split_path, sep="	")
    missing = [column for column in EXPECTED_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing columns in {split_path.name}: {missing}")
    frame = frame[EXPECTED_COLUMNS].copy()
    frame["split"] = split_name
    frame["tweet_text"] = frame["tweet_text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    frame = frame[frame["label"].isin(label2id)].copy()
    frame["label_id"] = frame["label"].map(label2id)
    frame["image_path"] = frame["image"].map(lambda value: str(resolve_image_path(value)))
    frame["token_length"] = frame["tweet_text"].str.split().str.len()
    frame = frame[frame["image_path"].map(lambda value: Path(value).exists())].reset_index(drop=True)
    if MAX_ROWS_PER_SPLIT is not None:
        frame = frame.head(MAX_ROWS_PER_SPLIT).copy()
    frame.to_csv(PROCESSED_DIR / f"{split_name}.csv", index=False)
    return frame


train_df = prepare_split(SPLIT_FILES["train"], "train")
val_df = prepare_split(SPLIT_FILES["dev"], "dev")
test_df = prepare_split(SPLIT_FILES["test"], "test")

split_sizes = pd.DataFrame({"split": ["train", "dev", "test"], "rows": [len(train_df), len(val_df), len(test_df)]})
label_summary = pd.concat([train_df, val_df, test_df], ignore_index=True).groupby(["split", "label"]).size().reset_index(name="count")
label_summary.to_csv(PROCESSED_DIR / "crisismmd_label_summary.csv", index=False)
text_length_summary = pd.concat([train_df, val_df, test_df], ignore_index=True).groupby("split")["token_length"].agg(["count", "mean", "median", "max"]).reset_index()
text_length_summary.to_csv(PROCESSED_DIR / "crisismmd_text_length_summary.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.barplot(data=split_sizes, x="split", y="rows", ax=axes[0], color="#17365d")
axes[0].set_title("Split sizes")
sns.countplot(data=pd.concat([train_df, val_df, test_df]), x="label", ax=axes[1], order=LABELS, color="#b38b2d")
axes[1].set_title("Label distribution")
axes[1].tick_params(axis="x", rotation=25)
sns.histplot(train_df["token_length"], bins=40, ax=axes[2], color="#2f7d6b")
axes[2].set_title("Tweet length distribution")
axes[2].set_xlabel("Tokens per tweet")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "crisismmd_eda_overview.png", dpi=180, bbox_inches="tight")
plt.close(fig)

display(split_sizes)
display(label_summary.groupby("label")["count"].sum().rename("total_count").to_frame())
display(text_length_summary)


In [ ]:
class MMDDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        return image, row["tweet_text"], int(row["label_id"])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, loss_value: float) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "loss": float(loss_value),
    }


def plot_history(history_df: pd.DataFrame, metric_col: str, title: str, output_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df[metric_col], marker="o", color="#17365d")
    axes[1].set_title(metric_col)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel(metric_col)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def run_epoch(model, loader, criterion, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            labels = batch.pop("labels").to(DEVICE)
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            if training:
                optimizer.zero_grad()
            logits = model(**batch)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            y_pred.append(preds)
            y_true.append(labels.detach().cpu().numpy())
            total_loss += float(loss.item()) * labels.size(0)
    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)
    avg_loss = total_loss / len(loader.dataset)
    return compute_metrics(y_true, y_pred, avg_loss), y_true, y_pred


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, monitor_key, ckpt_path, history_path):
    if ckpt_path.exists() and history_path.exists():
        print(f"Loading cached checkpoint: {ckpt_path.name}")
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        return pd.read_csv(history_path)
    rows = []
    best_score = -math.inf
    best_state = None
    patience_left = PATIENCE
    for epoch in range(1, epochs + 1):
        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer, scheduler)
        val_metrics, _, _ = run_epoch(model, val_loader, criterion)
        rows.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })
        if val_metrics[monitor_key] > best_score:
            best_score = val_metrics[monitor_key]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
    if best_state is None:
        raise RuntimeError("Training did not produce a valid checkpoint.")
    torch.save(best_state, ckpt_path)
    model.load_state_dict(best_state)
    history_df = pd.DataFrame(rows)
    history_df.to_csv(history_path, index=False)
    return history_df


train_set = MMDDataset(train_df)
val_set = MMDDataset(val_df)
test_set = MMDDataset(test_df)
class_weights = torch.tensor(train_df["label_id"].value_counts().sort_index().to_numpy(), dtype=torch.float32)
class_weights = (class_weights.sum() / class_weights).to(DEVICE)


In [ ]:
processor = AutoProcessor.from_pretrained(CLIP_CFG["model_name"])


def collate_batch(batch):
    images, texts, labels = zip(*batch)
    encoded = processor(text=list(texts), images=list(images), padding=True, truncation=True, max_length=CLIP_CFG["max_length"], return_tensors="pt")
    encoded["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoded


train_loader = DataLoader(train_set, batch_size=CLIP_CFG["batch_size"], shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(val_set, batch_size=CLIP_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(test_set, batch_size=CLIP_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)


class ClipClassifier(nn.Module):
    def __init__(self, model_name: str, num_classes: int, dropout: float = 0.2):
        super().__init__()
        self.backbone = CLIPModel.from_pretrained(model_name)
        dim = self.backbone.config.projection_dim
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(dim * 2, num_classes)

    def forward(self, input_ids, attention_mask, pixel_values, token_type_ids=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        features = torch.cat([outputs.image_embeds, outputs.text_embeds], dim=1)
        return self.head(self.dropout(features))


model = ClipClassifier(CLIP_CFG["model_name"], num_classes=len(LABELS), dropout=CLIP_CFG["dropout"]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=CLIP_CFG["lr"], weight_decay=CLIP_CFG["weight_decay"])
total_steps = len(train_loader) * CLIP_CFG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(total_steps // 10, 1), num_training_steps=total_steps)

ckpt_path = ARTIFACT_DIR / "crisismmd_clip_best.pt"
history_path = ARTIFACT_DIR / "crisismmd_clip_history.csv"
curve_path = ARTIFACT_DIR / "crisismmd_clip_history.png"

history_df = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, CLIP_CFG["epochs"], "macro_f1", ckpt_path, history_path)
plot_history(history_df, "val_macro_f1", "CLIP learning curves", curve_path)
clip_metrics, clip_y_true, clip_y_pred = run_epoch(model, test_loader, criterion)
clip_metrics


In [ ]:
processor = AutoProcessor.from_pretrained(VB_CFG["vision_name"])
tokenizer = AutoTokenizer.from_pretrained(VB_CFG["model_name"])


def collate_batch(batch):
    images, texts, labels = zip(*batch)
    image_inputs = processor(images=list(images), return_tensors="pt")
    text_inputs = tokenizer(list(texts), padding=True, truncation=True, max_length=VB_CFG["max_length"], return_tensors="pt")
    if "token_type_ids" not in text_inputs:
        text_inputs["token_type_ids"] = torch.zeros_like(text_inputs["input_ids"])
    text_inputs["pixel_values"] = image_inputs["pixel_values"]
    text_inputs["labels"] = torch.tensor(labels, dtype=torch.long)
    return text_inputs


train_loader = DataLoader(train_set, batch_size=VB_CFG["batch_size"], shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(val_set, batch_size=VB_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(test_set, batch_size=VB_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)


class VisualBertClassifier(nn.Module):
    def __init__(self, model_name: str, vision_name: str, num_classes: int, dropout: float = 0.2):
        super().__init__()
        self.vision = CLIPVisionModel.from_pretrained(vision_name)
        self.backbone = VisualBertModel.from_pretrained(model_name)
        clip_dim = self.vision.config.hidden_size
        visual_dim = self.backbone.config.visual_embedding_dim
        hidden_dim = self.backbone.config.hidden_size
        self.project = nn.Identity() if clip_dim == visual_dim else nn.Linear(clip_dim, visual_dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask, pixel_values, token_type_ids=None):
        vision_outputs = self.vision(pixel_values=pixel_values)
        visual_embeds = self.project(vision_outputs.last_hidden_state)
        visual_attention_mask = torch.ones(visual_embeds.shape[:2], dtype=torch.long, device=visual_embeds.device)
        visual_token_type_ids = torch.ones_like(visual_attention_mask)
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds,
            visual_attention_mask=visual_attention_mask,
            visual_token_type_ids=visual_token_type_ids,
            return_dict=True,
        )
        pooled = outputs.last_hidden_state[:, 0]
        return self.head(self.dropout(pooled))


model = VisualBertClassifier(VB_CFG["model_name"], VB_CFG["vision_name"], num_classes=len(LABELS), dropout=VB_CFG["dropout"]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=VB_CFG["lr"], weight_decay=VB_CFG["weight_decay"])
total_steps = len(train_loader) * VB_CFG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(total_steps // 10, 1), num_training_steps=total_steps)

ckpt_path = ARTIFACT_DIR / "crisismmd_visualbert_best.pt"
history_path = ARTIFACT_DIR / "crisismmd_visualbert_history.csv"
curve_path = ARTIFACT_DIR / "crisismmd_visualbert_history.png"

history_df = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, VB_CFG["epochs"], "macro_f1", ckpt_path, history_path)
plot_history(history_df, "val_macro_f1", "VisualBERT learning curves", curve_path)
visualbert_metrics, visualbert_y_true, visualbert_y_pred = run_epoch(model, test_loader, criterion)
visualbert_metrics


In [ ]:
clip_report = classification_report(clip_y_true, clip_y_pred, target_names=LABELS, output_dict=True, zero_division=0)
visualbert_report = classification_report(visualbert_y_true, visualbert_y_pred, target_names=LABELS, output_dict=True, zero_division=0)

comparison_df = pd.DataFrame([
    {"model": "CLIP", "accuracy": clip_metrics["accuracy"], "macro_f1": clip_metrics["macro_f1"]},
    {"model": "VisualBERT", "accuracy": visualbert_metrics["accuracy"], "macro_f1": visualbert_metrics["macro_f1"]},
])
comparison_df.to_csv(ARTIFACT_DIR / "crisismmd_model_comparison.csv", index=False)

per_label_df = pd.DataFrame({
    "label": LABELS,
    "clip_f1": [clip_report[label]["f1-score"] for label in LABELS],
    "visualbert_f1": [visualbert_report[label]["f1-score"] for label in LABELS],
})
per_label_df.to_csv(ARTIFACT_DIR / "crisismmd_per_label_f1.csv", index=False)

metrics_payload = {
    "sources": {
        "official_page": OFFICIAL_PAGE,
        "dataset": DATASET_URL,
        "agreed_split": AGREED_SPLIT_URL,
        "all_split": ALL_SPLIT_URL,
        "paper": PAPER_URL,
    },
    "label_to_id": label2id,
    "clip": clip_metrics,
    "visualbert": visualbert_metrics,
}
(ARTIFACT_DIR / "crisismmd_metrics_summary.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

clip_cm = confusion_matrix(clip_y_true, clip_y_pred, labels=list(range(len(LABELS))))
visualbert_cm = confusion_matrix(visualbert_y_true, visualbert_y_pred, labels=list(range(len(LABELS))))
(ARTIFACT_DIR / "crisismmd_confusion_matrices.json").write_text(json.dumps({"labels": LABELS, "clip": clip_cm.tolist(), "visualbert": visualbert_cm.tolist()}, indent=2), encoding="utf-8")

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(LABELS))
width = 0.35
ax.bar(x - width / 2, per_label_df["clip_f1"], width=width, label="CLIP", color="#17365d")
ax.bar(x + width / 2, per_label_df["visualbert_f1"], width=width, label="VisualBERT", color="#b38b2d")
ax.set_xticks(x)
ax.set_xticklabels(LABELS, rotation=25)
ax.set_ylabel("F1 score")
ax.set_title("Per-label F1 on the test split")
ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "crisismmd_per_label_f1.png", dpi=180, bbox_inches="tight")
plt.close(fig)


def save_confusion(matrix: np.ndarray, title: str, output_path: Path):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="viridis", xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


save_confusion(clip_cm, "CLIP confusion matrix", ARTIFACT_DIR / "crisismmd_confusion_clip.png")
save_confusion(visualbert_cm, "VisualBERT confusion matrix", ARTIFACT_DIR / "crisismmd_confusion_visualbert.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(clip_cm, annot=True, fmt="d", cmap="viridis", xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title("CLIP")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
sns.heatmap(visualbert_cm, annot=True, fmt="d", cmap="viridis", xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
axes[1].set_title("VisualBERT")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "crisismmd_confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.close(fig)

display(comparison_df)
display(per_label_df)
